# Building a RAG + AI Agents System — My Notes

okay so this notebook puts together everything we covered in class — RAG, web search with tavily, and multi-agent workflows using crewai. the idea is basically to build something similar to perplexity.ai where it decides whether to look stuff up from a document or search the web depending on what you ask.

i'm using groq for the LLM because it's free and honestly way faster than running anything locally.

**you'll need two API keys before running this:**
- groq: https://console.groq.com/keys
- tavily: https://app.tavily.com/home

both have free tiers, shouldn't need a credit card

## step 1 — install everything

heads up: this takes a minute the first time. crewai pulls in a bunch of dependencies

In [ ]:
%pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
%pip install langchain-groq
%pip install sentence-transformers
%pip install tavily-python
%pip install faiss-cpu
%pip install langchain langchain-openai
%pip install pypdf

## step 2 — imports

bringing in everything we need upfront so we don't run into import errors mid-run

In [ ]:
import os
from langchain_openai import ChatOpenAI
from crewai_tools import PDFSearchTool
from langchain_community.tools.tavily_search import TavilySearchResults
from crewai_tools import tool
from crewai import Crew, Task, Agent

# for the faiss stuff in the student tasks section
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chains import RetrievalQA

## step 3 — api keys

paste your keys below. i know hardcoding keys isn't great practice but for a local notebook it's fine — just don't push this to github lol

In [ ]:
# tavily needs this as an env variable, groq does too for some tools
GROQ_API_KEY = os.environ['GROQ_API_KEY']
TAVILY_API_KEY = os.environ['TAVILY_API_KEY']

# path to the pdf — keep it in the same folder as this notebook
PDF_PATH = 'doc.pdf'

## step 4 — setting up the LLM

using llama3 via groq. the trick here is that groq uses an openai-compatible API so we can just use ChatOpenAI and point it at groq's endpoint. temperature at 0.1 keeps answers consistent and not too random

In [ ]:
llm = ChatOpenAI(
    openai_api_base="https://api.groq.com/openai/v1",
    openai_api_key=GROQ_API_KEY,
    model_name="llama3-8b-8192",
    temperature=0.1,
    max_tokens=1000,
)

## step 5 — RAG tool (PDF knowledge base)

this is the core RAG part. PDFSearchTool takes our PDF, chunks it, embeds it using a huggingface model (BAAI/bge-small is lightweight and works well), and stores everything in chromadb so we can search it later.

note: first time you run this it downloads the embedding model (~130mb) so might take a bit

In [ ]:
rag_tool = PDFSearchTool(
    pdf=PDF_PATH,
    config=dict(
        llm=dict(
            provider="groq",
            config=dict(model="llama3-8b-8192"),
        ),
        embedder=dict(
            provider="huggingface",
            config=dict(model="BAAI/bge-small-en-v1.5"),
        ),
    )
)

In [ ]:
# quick sanity check — making sure the rag tool actually works before building agents on top of it
test = rag_tool.run("What is the main topic of this document?")
print(test[:400])

## step 6 — tavily web search

tavily is basically a search API built for LLMs — it returns clean structured results instead of raw HTML. k=3 means we get top 3 results per query

In [ ]:
web_search_tool = TavilySearchResults(k=3)

In [ ]:
# testing tavily
test_search = web_search_tool.run("latest AI models for healthcare 2024")
print(test_search[:400])

## step 7 — router tool

this decides whether a question should be answered from our pdf or from the web. simple keyword matching — if the question is about stuff in our document we go to the vectorstore, otherwise we search the web.

you can expand the keyword list depending on what your PDF is about

In [ ]:
@tool
def router_tool(question: str) -> str:
    """Routes a question to the appropriate source: vectorstore (PDF) or websearch."""
    pdf_keywords = [
        'sporo', 'ehr', 'clinical', 'chart', 'physician', 'patient',
        'summarization', 'hallucination', 'rag', 'retrieval', 'llm',
        'medical', 'healthcare', 'burnout', 'document'
    ]
    if any(kw in question.lower() for kw in pdf_keywords):
        return 'vectorstore'
    return 'websearch'


# test it out
print(router_tool.run("What does Sporo Health do?"))   # should be vectorstore
print(router_tool.run("Who won the FIFA world cup?"))  # should be websearch

## step 8 — defining the agents

this is where crewai comes in. we're building 5 agents that each handle one part of the pipeline:

1. **router** — decides where to look
2. **retriever** — actually fetches the info
3. **grader** — checks if what we got is relevant
4. **hallucination grader** — makes sure the answer isn't made up
5. **answer grader** — polishes and delivers the final answer

each agent has a role, goal, and backstory — the backstory is basically a system prompt that tells it how to behave

In [ ]:
Router_Agent = Agent(
    role='Router',
    goal='Route the user question to either the vectorstore or web search',
    backstory=(
        "You are an expert at deciding where to look for information. "
        "If the question is about the PDF document content (clinical summarization, EHR, "
        "Sporo Health, AI in healthcare, RAG, physician burnout), route to vectorstore. "
        "For everything else, route to web search."
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
)

In [ ]:
Retriever_Agent = Agent(
    role='Retriever',
    goal='Use the information from vectorstore or web to answer the question',
    backstory=(
        "You are a research assistant for question-answering. "
        "Use the retrieved context to give a clear, concise answer. "
        "Don't add info that isn't in the retrieved content."
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
)

In [ ]:
Grader_Agent = Agent(
    role='Relevance Grader',
    goal='Check if the retrieved content actually answers the question',
    backstory=(
        "You grade whether retrieved documents are relevant to the question. "
        "If the content contains keywords or ideas related to the question, it's relevant. "
        "Don't be too strict — partial relevance is fine."
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
)

In [ ]:
Hallucination_Grader = Agent(
    role='Hallucination Grader',
    goal='Make sure the answer is actually supported by the retrieved facts',
    backstory=(
        "You check whether answers are grounded in the retrieved documents. "
        "If the answer contains claims not supported by the facts, flag it. "
        "Be thorough — LLMs love making things up."
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
)

In [ ]:
Answer_Grader = Agent(
    role='Answer Grader',
    goal='Deliver a clean, useful final answer to the user',
    backstory=(
        "You produce the final answer. If previous agents confirmed the answer is good, "
        "return it clearly and concisely. If something went wrong upstream, "
        "fall back to a web search and try again."
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
)

## step 9 — tasks

tasks wire agents together. each task passes its output as context to the next one — that's what the `context=[]` parameter does. the `{question}` placeholder gets filled in when we call `crew.kickoff()`

In [ ]:
router_task = Task(
    description=(
        "Look at the question: {question}\n"
        "Decide whether it needs vectorstore search or web search.\n"
        "Reply with only one word: 'vectorstore' or 'websearch'. Nothing else."
    ),
    expected_output="Either 'vectorstore' or 'websearch'. No other text.",
    agent=Router_Agent,
    tools=[router_tool],
)

retriever_task = Task(
    description=(
        "Based on what the router decided, go fetch the answer to: {question}\n"
        "- router said 'websearch' → use web_search_tool\n"
        "- router said 'vectorstore' → use rag_tool"
    ),
    expected_output="Clear and concise text retrieved from the right source.",
    agent=Retriever_Agent,
    context=[router_task],
    tools=[rag_tool, web_search_tool],
)

grader_task = Task(
    description=(
        "Look at what the retriever found for: {question}\n"
        "Is it actually relevant to the question?"
    ),
    expected_output="Just 'yes' or 'no'. No explanation needed.",
    agent=Grader_Agent,
    context=[retriever_task],
)

hallucination_task = Task(
    description=(
        "For the question: {question}\n"
        "Check whether the answer from the grader step is actually supported by facts."
    ),
    expected_output="'yes' if the answer is factual and grounded, 'no' if it seems made up.",
    agent=Hallucination_Grader,
    context=[grader_task],
)

answer_task = Task(
    description=(
        "Final step for: {question}\n"
        "If hallucination grader said 'yes' → return a clean, concise final answer.\n"
        "If it said 'no' → do a web search and return what you find."
    ),
    expected_output=(
        "A clear, accurate answer to the question. "
        "If nothing was found, say: 'Sorry, couldn't find a valid answer for this.'"
    ),
    context=[hallucination_task],
    agent=Answer_Grader,
    tools=[web_search_tool],
)

## step 10 — assembling the crew

putting all agents and tasks together. crewai runs them sequentially — each agent gets the output of the previous one as context

In [ ]:
rag_crew = Crew(
    agents=[Router_Agent, Retriever_Agent, Grader_Agent, Hallucination_Grader, Answer_Grader],
    tasks=[router_task, retriever_task, grader_task, hallucination_task, answer_task],
    verbose=True,
)

## step 11 — testing it out

let's throw a couple questions at it. first one should pull from the pdf, second one should trigger a web search

In [ ]:
# question about the PDF content
result = rag_crew.kickoff(inputs={"question": "What are the main challenges of AI in clinical summarization?"})
print("\nAnswer:", result)

In [ ]:
# question that should trigger web search
result2 = rag_crew.kickoff(inputs={"question": "What are the latest LLM releases in 2024?"})
print("\nAnswer:", result2)

---
## bonus: researcher → writer → critic pipeline

this second crew mimics a real research workflow. researcher gathers info, writer turns it into a structured response, critic reviews and improves it. useful when you want more polished output than a single Q&A

In [ ]:
Researcher = Agent(
    role='Researcher',
    goal='Find thorough information on the given topic from both the PDF and the web',
    backstory=(
        "You're a detail-oriented researcher. You pull from the PDF knowledge base "
        "for domain-specific facts and use Tavily for anything current or supplementary. "
        "You compile everything into one coherent set of findings."
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
    tools=[rag_tool, web_search_tool],
)

Writer = Agent(
    role='Content Writer',
    goal='Turn research findings into a well-structured, readable response',
    backstory=(
        "You take raw research and write it up clearly. "
        "Your writing is organized, accurate, and easy to follow. "
        "You don't add fluff — just the important stuff."
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
)

Critic = Agent(
    role='Reviewer',
    goal='Check the written response for accuracy and completeness, then improve it',
    backstory=(
        "You review written responses with a critical eye. "
        "You catch inaccuracies, fill in gaps, and make sure everything actually "
        "answers the original question. You return the improved final version."
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
)

In [ ]:
research_task = Task(
    description=(
        "Research this topic thoroughly: {topic}\n"
        "Use the PDF knowledge base and web search. Compile all relevant findings."
    ),
    expected_output="A comprehensive set of facts and insights about the topic.",
    agent=Researcher,
)

write_task = Task(
    description=(
        "Write a detailed response about: {topic}\n"
        "Use the research findings. Include context, key facts, and a clear conclusion."
    ),
    expected_output="A well-structured response, roughly 3-5 paragraphs.",
    agent=Writer,
    context=[research_task],
)

critique_task = Task(
    description=(
        "Review the written piece about: {topic}\n"
        "Check for accuracy, completeness, and clarity. Return the improved version."
    ),
    expected_output="A polished, accurate final response on the topic.",
    agent=Critic,
    context=[write_task],
)

report_crew = Crew(
    agents=[Researcher, Writer, Critic],
    tasks=[research_task, write_task, critique_task],
    verbose=True,
)

In [ ]:
report_result = report_crew.kickoff(inputs={"topic": "How AI can help reduce physician burnout"})
print("\nFinal Report:\n", report_result)

---
## student tasks

### task 1 — explore the vectorstore

building a faiss index from scratch so we can run our own similarity searches. this is different from the crewai rag_tool which uses chromadb internally — here we have direct access to the vectorstore

In [ ]:
# load and chunk the pdf
loader  = PyPDFLoader(PDF_PATH)
pages   = loader.load()

# chunk size 500 with 50 token overlap works decently — smaller chunks = more precise retrieval
# but too small and you lose context. experiment with this
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks   = splitter.split_documents(pages)

print(f"loaded {len(pages)} pages → split into {len(chunks)} chunks")

In [ ]:
# embed and build the faiss index
embeddings  = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
vectorstore = FAISS.from_documents(chunks, embeddings)

print("faiss index built")

In [ ]:
# try a similarity search
query   = "What causes physician burnout?"
results = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(results, 1):
    print(f"\n--- result {i} (page {doc.metadata.get('page', '?')}) ---")
    print(doc.page_content[:300])

In [ ]:
# visualize the embeddings with pca — useful for seeing if chunks are clustering meaningfully
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# just using first 30 chunks otherwise it gets slow
sample_texts = [c.page_content for c in chunks[:30]]
vectors      = np.array(embeddings.embed_documents(sample_texts))

pca    = PCA(n_components=2)
coords = pca.fit_transform(vectors)

plt.figure(figsize=(8, 5))
plt.scatter(coords[:, 0], coords[:, 1], c=range(len(coords)), cmap='plasma', s=60)
plt.colorbar(label='chunk index')
plt.title("PCA of chunk embeddings")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.tight_layout()
plt.savefig("embeddings_pca.png", dpi=150)
plt.show()

### task 2 — retrieval QA chain

plugging the faiss vectorstore into langchain's RetrievalQA so we can do direct question answering with source tracking

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
qa_chain  = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True
)

In [ ]:
q      = "What percentage of physicians face burnout due to EHR workflows?"
output = qa_chain.invoke({"query": q})

print("Q:", q)
print("A:", output['result'])
print("\nsource pages:", [d.metadata.get('page') for d in output['source_documents']])

### task 3 — new agent: trend analyst

building a custom agent that just focuses on finding recent news. this is a good template for creating specialized agents — give it a focused role and one tool

In [ ]:
Trend_Analyst = Agent(
    role='Trend Analyst',
    goal='Find and summarize the latest developments on a given AI or tech topic',
    backstory=(
        "You're a trend spotter who keeps up with what's happening in AI and "
        "healthcare tech. You search the web, cut through the noise, and pull out "
        "the most relevant recent developments. You keep summaries short and focused."
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
    tools=[web_search_tool],
)

trend_task = Task(
    description="Search and summarize recent news on: {trend_topic}",
    expected_output="5 recent developments as short bullet points.",
    agent=Trend_Analyst,
)

trend_crew   = Crew(agents=[Trend_Analyst], tasks=[trend_task], verbose=True)
trend_result = trend_crew.kickoff(inputs={"trend_topic": "AI agents in healthcare 2024"})

print("\nTrends:\n", trend_result)

### task 4 — report generator

agent that produces a structured markdown report and saves it to a file. could easily extend this to export as pdf or word doc

In [ ]:
from datetime import datetime

Report_Generator = Agent(
    role='Report Generator',
    goal='Create a structured markdown report on a given topic',
    backstory=(
        "You write thorough, well-structured research reports in Markdown. "
        "You gather info from the PDF knowledge base and web, then organize it "
        "into clear sections. No filler content — only what's useful."
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
    tools=[rag_tool, web_search_tool],
)

report_task = Task(
    description=(
        "Write a structured markdown report on: {report_topic}\n\n"
        "Include these sections:\n"
        "- Executive Summary\n"
        "- Key Findings (3-5 bullet points)\n"
        "- Challenges and Limitations\n"
        "- Future Outlook\n"
        "- Conclusion\n\n"
        "Pull from the PDF knowledge base and web search."
    ),
    expected_output="Full markdown report with all sections filled in.",
    agent=Report_Generator,
)

report_gen_crew = Crew(agents=[Report_Generator], tasks=[report_task], verbose=True)

In [ ]:
topic      = "AI in clinical summarization: where we are and where it's going"
report_out = report_gen_crew.kickoff(inputs={"report_topic": topic})

# save to a file
fname = f"report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.md"
with open(fname, "w") as f:
    f.write(f"# {topic}\n")
    f.write(f"*generated on {datetime.now().strftime('%d %b %Y, %H:%M')}*\n\n")
    f.write(str(report_out))

print(f"saved to {fname}")
print("\npreview:")
print(str(report_out)[:500])

### bonus — interactive Q&A loop

wrapping everything in a simple loop so you can ask questions without rerunning cells. type 'quit' to stop

In [ ]:
print("ask me anything (based on the pdf or web search)")
print("type 'quit' to exit\n")

while True:
    q = input("you: ").strip()
    if q.lower() in ['quit', 'exit', 'q', '']:
        print("bye!")
        break
    ans = rag_crew.kickoff(inputs={"question": q})
    print("\nbot:", ans)
    print("-" * 50)

---
## quick summary of what we built

| component | what it does |
|---|---|
| groq + llama3 | fast cloud LLM, no GPU needed |
| PDFSearchTool | RAG over doc.pdf using chromadb |
| FAISS vectorstore | direct similarity search over chunks |
| Tavily | real-time web search |
| router agent | routes to vectorstore vs web |
| retriever agent | fetches the actual info |
| grader agent | checks relevance |
| hallucination grader | catches made-up answers |
| answer grader | final clean answer |
| researcher/writer/critic | full report generation pipeline |
| trend analyst | real-time news summaries |
| report generator | exports markdown reports |

things to try next: swap FAISS for Chroma, try different chunk sizes, replace groq with ollama for fully local inference, wrap this in a streamlit app